In [1]:
import whisper

In [2]:
import string
import numpy as np
def text_normalizer(text):
    text = text.upper()
    return text.translate(str.maketrans('', '', string.punctuation))
def compute_wer(hyp_sentence="",ref_sentence=""):
    """
    Inputs: 
    hyp_sentence: str- Sentence of text from the ASR Hypothesis
    ref_sentence: str-Sentence of text from the Ground Truth Reference
    Returns:
    wer_score: float- WER Score as a floating point number rounded to two decimal places       
    """
    ### 2024.11.15
    hyp_sentence = text_normalizer(hyp_sentence)
    ref_sentence = text_normalizer(ref_sentence)

    ## Fill your code here
    hyp_word = hyp_sentence.split()
    ref_word = ref_sentence.split()

    m = len(hyp_word)
    n = len(ref_word)

    w_table = np.zeros([m + 1, n + 1])
    w_table[0, :] = np.arange(n + 1)
    w_table[:, 0] = np.arange(m + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if hyp_word[i - 1] == ref_word[j - 1]:
                w_table[i, j] = w_table[i - 1, j - 1]
            else:
                w_table[i, j] = 1 + min(w_table[i - 1, j], w_table[i, j - 1], w_table[i - 1, j - 1])

    score = w_table[m, n] / n
    return score * 100

In [3]:
def compute_cer(hyp_sentence="", ref_sentence=""):

    hyp_sentence = text_normalizer(hyp_sentence)
    ref_sentence = text_normalizer(ref_sentence)

    ## 将句子转换为字符列表，而不是按词拆分
    hyp_char = list(hyp_sentence)
    ref_char = list(ref_sentence)

    m = len(hyp_char)
    n = len(ref_char)

    c_table = np.zeros([m + 1, n + 1])
    c_table[0, :] = np.arange(n + 1)
    c_table[:, 0] = np.arange(m + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if hyp_char[i - 1] == ref_char[j - 1]:
                c_table[i, j] = c_table[i - 1, j - 1]
            else:
                c_table[i, j] = 1 + min(c_table[i - 1, j], c_table[i, j - 1], c_table[i - 1, j - 1])

    score = c_table[m, n] / n if n > 0 else 0
    return score * 100

In [4]:
# cache_path: ~/.cache

In [5]:
# whisper.load_model("medium.en")

In [6]:
# whisper.load_model("small.en")

In [7]:
# whisper.load_model("base.en")

In [4]:
import os
def search_wavs(rootdir):
    paths=[]
    for root, dirs, files in os.walk(rootdir):
        for file in files:
            if file.endswith(".wav"):
                paths.append(os.path.join(root,file))
    
    return paths

In [5]:
asrmodel=whisper.load_model("medium.en")

In [23]:
asrmodel=whisper.load_model("turbo") #ssc

In [9]:
result = asrmodel.transcribe("mos/oracle/p238/p238_001.wav")
result["text"]

' Please call Stella.'

In [10]:
result1 = asrmodel.transcribe("mos/outputs_diffvc/p238/p238_p241_001.wav")
result1["text"]

' Please call still.'

In [11]:
result1 = asrmodel.transcribe("mos/outputs_diffvcplus/p238/p238_p241_001.wav")
result1["text"]

' Please call Stella.'

In [7]:
result = asrmodel.transcribe('/home/huangf79/projects/DiffVC/es_tts_t2e/121/121_121726_000025_000001.wav')
result["text"]

' once held by Hobson and Dewey, now carried by Mother Eddie and Brother Dewey.'

In [6]:
compute_wer('     Please call still.','Please call Stella.')

33.33333333333333

In [5]:
utt_map = {
    '001': 'Please call Stella.',
    '002': 'Ask her to bring these things with her from the store.',
    '003': 'Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob.',
    '004': 'We also need a small plastic snake and a big toy frog for the kids.',
    '005': 'She can scoop these things into three red bags, and we will go meet her Wednesday at the train station.'
}
def get_transcript(path):
    utt = path.split('_')[-1][:3]
    return utt_map[utt]
get_transcript("mos/outputs_diffvc/p238/p238_p241_001.wav")

'Please call Stella.'

In [6]:
def get_oraclepath(path,oracledir='mos/oracle'):
    name=path.split('/')[-1]
    info=name.split('_')
    sourcespk=info[0]
    utt=info[-1][:3]
    return f'{oracledir}/{sourcespk}/{sourcespk}_{utt}.wav'
get_oraclepath("mos/outputs_diffvc/p238/p238_p241_001.wav")

'mos/oracle/p238/p238_001.wav'

In [7]:
import numpy as np
from tqdm import tqdm
def batch_wer(dir, mode='transcript'):
    assert mode in ['transcript','oracle']
    wers=[]
    for wav in tqdm(search_wavs(dir)):
        hat = asrmodel.transcribe(wav)["text"]
        if mode == 'transcript':
            gt = get_transcript(wav)
        else:
            oracle = get_oraclepath(wav)
            gt = asrmodel.transcribe(oracle)["text"]
        wer = compute_wer(hyp_sentence=hat,ref_sentence=gt)
        wers.append(wer)
    return np.mean(wers), wers 

In [32]:
# small.en
# wer, wers = batch_wer('mos/oracle/')
# print(wer)

100%|██████████| 50/50 [00:29<00:00,  1.67it/s]

4.766666666666667


In [16]:
# medium.en
# wer, wers = batch_wer('mos/oracle/')
# print(wer)

100%|██████████| 50/50 [00:54<00:00,  1.09s/it]

2.5666666666666664


In [22]:
# wer, wers = batch_wer('mos/oracle/',mode='oracle')
# print(wer)

100%|██████████| 50/50 [01:11<00:00,  1.44s/it]

0.0


In [16]:
# small.en
# wer, wers = batch_wer('mos/outputs_diffvc')
# print(wer)

100%|██████████| 450/450 [05:39<00:00,  1.33it/s]

58.46902356902357


In [17]:
# medium.en
# wer, wers = batch_wer('mos/outputs_diffvc')
# print(wer)

100%|██████████| 450/450 [09:47<00:00,  1.31s/it]

56.2956228956229


In [17]:
# small.en
# wer, wers = batch_wer('mos/outputs_diffvcplus')
# print(wer)

100%|██████████| 450/450 [04:29<00:00,  1.67it/s]

5.742424242424242


In [20]:
# medium.en
# wer, wers = batch_wer('mos/outputs_diffvcplus')
# print(wer)

100%|██████████| 450/450 [08:09<00:00,  1.09s/it]

2.7740740740740737


In [16]:
wers_t={}
wers_o={}

for dir in ['mos/oracle/',
            'mos/outputs_diffvc/',
            'mos/outputs_diffvcplus/',
            'mos/outputs_diffvcp_light/',
            'mos/outputs_diffvcp_ctrl/']:
    wer1, _ = batch_wer(dir)
    wers_t[dir]=wer1
    wer2, _ = batch_wer(dir,mode = 'oracle')
    wers_o[dir]=wer2
    print(wer1, wer2)

print(wers_t, '\n',  wers_o)

100%|██████████| 50/50 [01:47<00:00,  2.15s/it]


2.5666666666666664 0.0


100%|██████████| 450/450 [17:50<00:00,  2.38s/it]


55.513468013468014 55.72183742183743


100%|██████████| 450/450 [16:02<00:00,  2.14s/it]


2.7740740740740737 2.0402116402116404


100%|██████████| 450/450 [16:04<00:00,  2.14s/it]


3.230976430976431 2.308225108225108


100%|██████████| 450/450 [16:04<00:00,  2.14s/it]

3.1875420875420875 2.3647907647907647
{'mos/oracle/': 2.5666666666666664, 'mos/outputs_diffvc/': 55.513468013468014, 'mos/outputs_diffvcplus/': 2.7740740740740737, 'mos/outputs_diffvcp_light/': 3.230976430976431, 'mos/outputs_diffvcp_ctrl/': 3.1875420875420875} 
 {'mos/oracle/': 0.0, 'mos/outputs_diffvc/': 55.72183742183743, 'mos/outputs_diffvcplus/': 2.0402116402116404, 'mos/outputs_diffvcp_light/': 2.308225108225108, 'mos/outputs_diffvcp_ctrl/': 2.3647907647907647}


In [19]:
wer1, _ = batch_wer('mos/outputs_diffvcp_ctrl/')
wer2, _ = batch_wer('mos/outputs_diffvcp_ctrl/', mode = 'oracle')
print(wer1,wer2)

100%|██████████| 450/450 [16:10<00:00,  2.16s/it]

3.1875420875420875 2.3647907647907647


In [20]:
# medium.en
# wer1, _ = batch_wer('outputs_diffvc_official')
# print(wer1)

100%|██████████| 450/450 [10:07<00:00,  1.35s/it]

63.815488215488216


In [11]:
wer1, _ = batch_wer('./outputs_diffvcp/')
wer2, _ = batch_wer('./outputs_diffvcp/', mode = 'oracle')


100%|██████████| 50/50 [02:17<00:00,  2.75s/it]

2.945454545454545 2.412121212121212


In [14]:
wer1, _ = batch_wer('./outputs_diffvcpl/')
wer2, _ = batch_wer('./outputs_diffvcpl/', mode = 'oracle')
print(wer1,wer2)

100%|██████████| 50/50 [02:12<00:00,  2.66s/it]

3.2121212121212124 2.6073593073593075


In [24]:
wer1, _ = batch_wer('./outputs_diffvcpd/')
wer2, _ = batch_wer('./outputs_diffvcpd/', mode = 'oracle')
print(wer1,wer2)

100%|██████████| 50/50 [02:19<00:00,  2.79s/it]

3.03030303030303 2.420779220779221


In [25]:
wer1, _ = batch_wer('./outputs_diffvcpd_uc')
wer2, _ = batch_wer('./outputs_diffvcpd_uc', mode = 'oracle')
print(wer1,wer2)

100%|██████████| 50/50 [01:49<00:00,  2.19s/it]

3.63030303030303 2.792207792207792


In [12]:
wer1, _ = batch_wer('mos/outputs_diffvc_official')
wer2, _ = batch_wer('mos/outputs_diffvc_official', mode = 'oracle')
wer1,wer2

100%|██████████| 450/450 [17:48<00:00,  2.37s/it]


(68.70067340067341, 64.18648388648388)

In [12]:
wer1, _ = batch_wer('./outputs_diffvcp_light_streaming')
wer2, _ = batch_wer('./outputs_diffvcp_light_streaming', mode = 'oracle')
wer1,wer2

100%|██████████| 450/450 [16:11<00:00,  2.16s/it]


(3.059259259259259, 2.3957671957671955)

: 

In [13]:
wer1, _ = batch_wer('./outputs_test')
wer2, _ = batch_wer('./outputs_test', mode = 'oracle')
wer1,wer2

100%|██████████| 450/450 [18:17<00:00,  2.44s/it]


(2.8777777777777778, 2.0216931216931218)

In [8]:
wer1, _ = batch_wer('./outputs_eg')
wer2, _ = batch_wer('./outputs_eg', mode = 'oracle')
wer1,wer2

100%|██████████| 50/50 [01:49<00:00,  2.20s/it]


(2.2, 2.457142857142857)

In [9]:
wer1, _ = batch_wer('./outputs_uc')
wer2, _ = batch_wer('./outputs_uc', mode = 'oracle')
wer1,wer2

100%|██████████| 50/50 [01:49<00:00,  2.20s/it]


(3.833333333333333, 1.8571428571428572)

# embspeech

## TTS

In [8]:
import numpy as np

In [9]:
lines = np.load('/home/huangf79/projects/UnitSpeech/notebooks/tts_info.npy').tolist()
lines

[['121_127105_000043_000004',
  'He was handsome and bold and pleasant, offhand and gay and kind.',
  'He was handsome and bold and pleasant, offhand and gay and kind.'],
 ['121_127105_000040_000000',
  '"With this outbreak at last."',
  '"With this outbreak at last."'],
 ['121_127105_000012_000001',
  'He passed his hand over his eyes, made a little wincing grimace.',
  'He passed his hand over his eyes, made a little wincing grimace.'],
 ['121_127105_000016_000002',
  'I shall have to send to town." There was a unanimous groan at this, and much reproach; after which, in his preoccupied way, he explained.',
  'I shall have to send to town." There was a unanimous groan at this, and much reproach; after which, in his preoccupied way, he explained.'],
 ['121_127105_000008_000000',
  '"I quite agree--in regard to Griffin\'s ghost, or whatever it was--that its appearing first to the little boy, at so tender an age, adds a particular touch.',
  '"I quite agree-in regard to Griffin\'s ghost,

In [10]:
from tqdm import tqdm

In [23]:
## gt
dir = '/home/huangf79/projects/Grad-TTS/LibriTTS/test-clean'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    book = filename.split('_')[1] #
    wavpath = f'{dir}/{spk}/{book}/{filename}.wav' #
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:20<00:00,  1.25it/s]


(4.237600659653286, 11.146520902841374, 3.661453286980735, 9.241361696122203)

In [ ]:
dir = 'es_tts_t2e' #√
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:20<00:00,  1.25it/s]


(5.0796584343027815, 11.6767324583847, 3.92959244051315, 9.28937507903541)

In [12]:
dir = 'es_tts_t2e-1'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [00:49<00:00,  2.01it/s]


(5.109299570152935, 14.872631156325577, 3.592080784091646, 9.227914909348263)

In [11]:
dir = 'es_tts_t2e-16k'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:24<00:00,  1.18it/s]


(5.591235348767627, 12.123576576692988, 4.035446814523398, 9.327071234259474)

In [25]:
dir = 'es_tts_t2e_75'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:19<00:00,  1.26it/s]


(6.202074163292379, 15.1733740397376, 4.0775783285492775, 9.548676235350866)

In [26]:
dir = 'es_tts_t2e_50'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:19<00:00,  1.25it/s]


(7.082884904340191, 14.016611290876327, 5.352803547471007, 12.672043342510275)

In [16]:
dir = 'es_tts_t2e_25'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:30<00:00,  1.11it/s]


(8.29531473075576, 15.952069843470273, 5.851227631246965, 14.771699578074719)

In [17]:
dir = 'es_tts_t2e_05'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:30<00:00,  1.11it/s]


(10.064568198692426, 16.419996129350853, 6.9989325723089, 14.910348037892787)

In [ ]:
###

In [28]:
# us zs
dir = '/home/huangf79/projects/UnitSpeech/notebooks/uszs'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:21<00:00,  1.23it/s]


(5.249753034821431, 12.72836288089328, 4.2764216703749165, 10.228834222089242)

In [29]:
dir = '/home/huangf79/projects/UnitSpeech/notebooks/usft'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:26<00:00,  1.16it/s]


(4.814300357824427, 11.763840213748102, 3.806372637563915, 9.282964334736786)

In [31]:
# yourtts
dir = '/home/huangf79/projects/UnitSpeech/yourtts_test/tts_zs'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:21<00:00,  1.23it/s]


(11.77367062450837, 19.614909993792455, 7.587462961528645, 12.001611531067343)

In [14]:
dir = '/home/huangf79/projects/xe_test/tts_zs'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:15<00:00,  1.32it/s]


(33.68254992379353, 23.379052106947146, 20.348322809817184, 15.917210619900981)

In [12]:
dir = '/home/huangf79/projects/CosyVoice/cosy_test/tts_zs'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/{spk}/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:16<00:00,  1.31it/s]


(7.337909731666005, 16.2054185892728, 6.175645094461917, 14.503931279508377)

In [11]:
from tqdm import tqdm
dir = '/home/huangf79/projects/vits_test/lj'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/spk0/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:18<00:00,  1.27it/s]


(5.182894610022686, 11.921743959056768, 3.9233700978198147, 9.463224749607622)

In [14]:
wers.sort()
cers.sort()
np.mean(wers[10:]), np.std(wers[10:]), np.mean(cers[10:]), np.std(cers[10:])

(5.758771788914094, 12.433970863851412, 4.30700341141306, 9.90096938451133)

In [18]:
dir = '/home/huangf79/projects/vits_test/vctk'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/p245/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:17<00:00,  1.29it/s]


(18.413291669021113, 24.442747602353993, 11.46918155461078, 18.495251401048645)

In [19]:
dir = '/home/huangf79/projects/vits_test/vctk'
# wers = []
# cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/p270/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:15<00:00,  1.33it/s]


(17.905777665667447, 24.69584813530657, 11.359956777196057, 18.167030312769963)

In [21]:
len(wers)

200

In [43]:
dir = '/home/huangf79/projects/DiffVC/es_tts_t2e_lj'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/spk0/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:17<00:00,  1.29it/s]


(5.456397498500864, 12.202610838913568, 4.053104700678416, 9.385923308647712)

In [23]:
dir = '/home/huangf79/projects/DiffVC/es_tts_t2e_vctk'
wers = []
cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/p245/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:18<00:00,  1.28it/s]


(10.089532707612237, 37.473129787519255, 7.944162620601222, 41.868987090935235)

In [24]:
dir = '/home/huangf79/projects/DiffVC/es_tts_t2e_vctk'
# wers = []
# cers = []
for filename, _, gt in tqdm(lines):
    spk = filename.split('_')[0]
    wavpath = f'{dir}/p270/{filename}.wav'
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers),len(wers)


100%|██████████| 100/100 [01:17<00:00,  1.30it/s]


(8.25299210857164,
 28.723793157903835,
 6.456030894776724,
 31.137062210043307,
 200)

## VC

In [26]:
def filepath2gt(filepath):
    filename = os.path.basename(filepath)
    dir = '/home/huangf79/projects/Grad-TTS/LibriTTS/test-clean'
    spk = filename.split('_')[0]
    book = filename.split('_')[1] 
    gtname = filename.split('_to_')[0] + '.normalized.txt'
    gtpath = f'{dir}/{spk}/{book}/{gtname}'
    with open(gtpath, 'r') as ff:
        gt = ff.readlines()[0].strip()
    return gt

In [52]:
with open('/home/huangf79/projects/UnitSpeech/notebooks/src-tgt.txt','r') as ff:
    srcs = ff.readlines()
    srcs = [line.split('|')[0] for line in srcs]
wers = []
cers = []
for wavpath in tqdm(srcs):
    gtpath = wavpath.replace('.wav', '.normalized.txt')
    with open(gtpath, 'r') as ff:
        gt = ff.readlines()[0].strip()
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:23<00:00,  1.20it/s]


(4.053588045482606, 6.946915157535937, 2.2454291519117606, 3.9829144300274626)

In [ ]:
dir = 'es_vc'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:23<00:00,  1.20it/s]


(4.612754578025813, 6.977669836467781, 2.7715930670260978, 4.333911561321648)

In [22]:
dir = 'es_vc-16k'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:51<00:00,  1.12s/it]


(4.567500768226998, 7.110146343228762, 2.668239108733736, 4.22732427824277)

In [46]:
dir = '/home/huangf79/projects/UnitSpeech/notebooks/uszs_vc'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:24<00:00,  1.18it/s]


(6.597766178467594, 8.784329322445204, 3.722578851700183, 5.061647562360005)

In [47]:
dir = '/home/huangf79/projects/UnitSpeech/notebooks/usft_vc'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:24<00:00,  1.18it/s]


(6.118079862384007, 7.94691129948399, 3.4251230579230607, 4.65517570530126)

In [48]:
dir = '/home/huangf79/projects/UnitSpeech/yourtts_test/vc_zs'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:23<00:00,  1.20it/s]


(6.584953555528022, 7.591687040390412, 3.9868620357783664, 5.007776736554334)

In [ ]:
### es re

In [27]:
dir = '/home/huangf79/projects/CosyVoice/cosy_test/vc_zs'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:21<00:00,  1.22it/s]


(4.997486365946528, 7.559076772096502, 2.8193192257094495, 4.499520813445114)

In [28]:
dir = '/home/huangf79/projects/xe_test/vc_zs'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:19<00:00,  1.25it/s]


(10.32345433962764, 10.927664515494675, 5.805147042329822, 6.285857183649695)

In [31]:
dir = '/home/huangf79/projects/FreeVC/freevcs_test_409'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:19<00:00,  1.25it/s]


(4.884728373494918, 7.713156400494111, 2.9764972010965582, 4.726809290368125)

In [29]:
dir = '/home/huangf79/projects/DiffVC/diffvc_test'
wers = []
cers = []
for wavpath in tqdm(search_wavs(dir)):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 100/100 [01:35<00:00,  1.04it/s]


(37.349165709885675, 24.280126688809116, 21.8241863141652, 17.10908028343918)

In [30]:
wers

[61.53846153846154,
 15.789473684210526,
 47.61904761904761,
 20.689655172413794,
 14.285714285714285,
 31.818181818181817,
 14.285714285714285,
 46.666666666666664,
 100.0,
 33.33333333333333,
 52.17391304347826,
 35.714285714285715,
 45.45454545454545,
 40.0,
 40.0,
 76.92307692307693,
 42.857142857142854,
 32.25806451612903,
 38.095238095238095,
 35.714285714285715,
 0.0,
 60.0,
 47.368421052631575,
 42.857142857142854,
 56.52173913043478,
 17.24137931034483,
 36.84210526315789,
 19.047619047619047,
 56.75675675675676,
 64.70588235294117,
 25.806451612903224,
 52.63157894736842,
 5.263157894736842,
 11.11111111111111,
 51.85185185185185,
 27.77777777777778,
 3.7037037037037033,
 100.0,
 106.66666666666667,
 52.94117647058824,
 23.333333333333332,
 100.0,
 16.666666666666664,
 31.57894736842105,
 8.333333333333332,
 16.666666666666664,
 0.0,
 24.0,
 30.0,
 8.0,
 21.73913043478261,
 38.88888888888889,
 52.94117647058824,
 13.043478260869565,
 13.043478260869565,
 20.833333333333336,
 

## SSC

In [13]:
def filepath2gt(filepath):
    filename = os.path.basename(filepath).split('.')[0]
    dir = '/home/huangf79/projects/DISSC/results/vctk/dissc_b'
    spk = filepath.split('/')[-2]
    gtpath = f'{dir}/{spk}/{filename}.txt'
    with open(gtpath, 'r') as ff:
        gt = ff.readlines()[0].strip()
    # print(filename,gt)
    return gt

In [7]:
from tqdm import tqdm

In [37]:
dir = 'es_u2e'
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 120/120 [00:47<00:00,  2.51it/s]


(14.410462086932675, 16.054769207164547, 8.81065302271912, 8.264260980005725)

In [16]:
dir = 'es_u2e-16k'
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 120/120 [00:52<00:00,  2.27it/s]


(14.332130696101284, 16.034099898850048, 8.514693473573319, 7.992733071227284)

In [26]:
dir = 'es_u2e-1'
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 120/120 [00:47<00:00,  2.55it/s]


(14.17047454179807, 15.218239486807219, 8.651868323783638, 7.656339027842939)

In [27]:
dir = '/home/huangf79/projects/DISSC/results/vctk/dissc_b'
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 120/120 [00:46<00:00,  2.60it/s]


(16.788015905662963, 16.4517033818669, 9.73190319276841, 7.880617414605802)

In [30]:
dir = '/home/huangf79/projects/DISSC/results/vctk/orig'
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
# wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt_path = wavpath.replace('.wav','.txt')
    with open(gt_path,'r') as ff:
        gt = ff.readlines()[0].strip()
    # gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 40/40 [00:15<00:00,  2.57it/s]


(1.0256410256410255, 2.715104946736842, 2.281423635314652, 1.5386902487773695)

In [19]:
# medium
dir = '/home/huangf79/projects/DISSC/results/vctk/orig' 
wers = []
cers = []
wavpaths = search_wavs(dir)
wavpaths = [wavpath for wavpath in wavpaths if int(os.path.basename(wavpath).split('.')[0].split('_')[1])<=10]
# wavpaths = [wavpath for wavpath in wavpaths if wavpath.split('/')[-2] != wavpath.split('/')[-1].split('_')[0]]
for wavpath in tqdm(wavpaths):
    gt_path = wavpath.replace('.wav','.txt')
    with open(gt_path,'r') as ff:
        gt = ff.readlines()[0].strip()
    # gt = filepath2gt(wavpath)
    hat = asrmodel.transcribe(wavpath)["text"]
    wers.append(compute_wer(hat, gt))
    cers.append(compute_cer(hat, gt))
np.mean(wers), np.std(wers), np.mean(cers), np.std(cers)

100%|██████████| 40/40 [00:27<00:00,  1.44it/s]


(0.41666666666666663, 1.816207893141947, 2.126151608352688, 1.3918885135709367)